# 09 - Graph RAG vs. fine-tune comparison


## Goal

Run the graph-only RAG answer against a placeholder for a fine-tuned model and discuss *when each approach wins*. The fine-tune side is a stub today; flip the switch once you have real weights from notebook 07.


## Prerequisites

- Notebooks 02 and 08 read.
- No GPU. No weights. The fine-tune column is a comment block, not a real call.


## Environment bootstrap

This cell makes the notebook portable between a local checkout and Colab.

- **Local**: when the notebook lives inside the repo, we add the repo root to `sys.path`
  so the `src` package imports cleanly.
- **Colab**: the import will fail with `ModuleNotFoundError`. We catch that and print a
  one-line reminder showing the `git clone` the learner should run. We deliberately do
  **not** execute the clone for them — the lab policy is *recipes only, no auto-downloads*.


In [ ]:
import sys
from pathlib import Path

try:
    # Local checkout: walk up from the notebook to the repo root.
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "src" / "common" / "paths.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            break
    from src.common.paths import REPO_ROOT, MINI_REPO, ensure_dirs
    ensure_dirs()
    print(f"repo root: {REPO_ROOT}")
    print(f"mini repo: {MINI_REPO}")
except ModuleNotFoundError:
    print("`src` not importable. If you are on Colab, run this in a separate cell:")
    print("    !git clone https://example.invalid/kde_ontology_slm_lab.git")
    print("    %cd kde_ontology_slm_lab")
    print("Then re-run this cell. We will not auto-clone for you (lab policy:")
    print("recipes only, no auto-downloads).")


## 1. Build the graph and pick a question


In [ ]:
from src.common.paths import MINI_REPO
from src.repo_ingest.scanner import scan
from src.repo_ingest.cmake_reader import read_cmake
from src.repo_ingest.cpp_reader import read_cpp
from src.repo_ingest.qml_reader import read_qml
from src.repo_ingest.dbus_reader import read_dbus
from src.repo_ingest.kconfig_reader import read_kconfig
from src.repo_ingest.desktop_file_reader import read_desktop
from src.repo_ingest.log_reader import read_log
from src.ontology.extractor import (
    ExtractionBundle, from_cmake, from_cpp, from_qml, from_dbus,
    from_kconfig, from_desktop, from_log,
)
from src.ontology.schema import Entity
from src.common.ids import make_id
from src.graph.builder import build_graph

rep = scan(MINI_REPO)
b = ExtractionBundle()
rid = b.add_entity(Entity(id=make_id('Repository', MINI_REPO.name),
                          type='Repository', name=MINI_REPO.name,
                          source_path=str(MINI_REPO)))
for sf in rep.by_kind('cmake'): from_cmake(b, read_cmake(sf.path), rid)
for sf in rep.by_kind('cpp_header') + rep.by_kind('cpp_source'): from_cpp(b, read_cpp(sf.path))
for sf in rep.by_kind('qml'): from_qml(b, read_qml(sf.path))
for sf in rep.by_kind('dbus'): from_dbus(b, read_dbus(sf.path))
for sf in rep.by_kind('kconfig'): from_kconfig(b, read_kconfig(sf.path))
for sf in rep.by_kind('desktop'): from_desktop(b, read_desktop(sf.path))
for sf in rep.by_kind('log'): from_log(b, read_log(sf.path))
g = build_graph(b)

question = 'Which signals does the KFileSearcher class emit?'
print('Q:', question)


## 2. Column A - Graph-only RAG

Deterministic, evidence-cited, no model. See `src/rag/answer_with_evidence.py`.


In [ ]:
from src.rag.answer_with_evidence import answer

rag_answer = answer(g, question, k=6)
print(rag_answer.text)


## 3. Column B - Fine-tuned model (stub)

Real call would look like the code in the comment block below. Today this cell only prints a placeholder so the notebook stays runnable without weights.

```python
# Hypothetical fine-tuned-model invocation. Requires:
#   - the [train] extra installed
#   - a base model whose CHANGE_ME has been replaced in configs/models.yaml
#   - a LoRA adapter trained per notebook 07
#
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from peft import PeftModel
# base = AutoModelForCausalLM.from_pretrained('Qwen/<real-id>')
# tok = AutoTokenizer.from_pretrained('Qwen/<real-id>')
# model = PeftModel.from_pretrained(base, 'artifacts/training_runs/v0_qwen_sft')
# inputs = tok(question, return_tensors='pt')
# ft_answer_text = tok.decode(model.generate(**inputs, max_new_tokens=256)[0])
```


In [ ]:
ft_answer_text = (
    '[stub] A trained KDE-SLM would answer here, ideally citing the same evidence\n'
    'the RAG path cites in column A. Today this is a placeholder string so the\n'
    'notebook runs without GPU/weights. Train via notebook 07, then call the\n'
    'commented snippet above and assign its output to ft_answer_text.'
)
print(ft_answer_text)


## 4. Side-by-side comparison


In [ ]:
print('=' * 70)
print('Graph-only RAG'.center(70))
print('=' * 70)
print(rag_answer.text)
print()
print('=' * 70)
print('Fine-tuned model (stub)'.center(70))
print('=' * 70)
print(ft_answer_text)


## 5. When each wins

| Scenario | RAG | Fine-tune |
|---|---|---|
| New entity added yesterday | wins (just re-extract) | needs re-training |
| Question phrasing far from any rule template | weaker | wins (generalises) |
| Strict citation required | wins (cites by construction) | needs grounding head |
| Offline, no GPU | wins | impossible |
| Conversational rephrasing | weaker | wins |
| Latency budget 50 ms | wins | depends on model size |

Practical recipe: **hybrid**. Use the graph retriever to fetch evidence; pass the evidence into the fine-tuned model's prompt; let the model phrase the answer. The lab scaffolds this in `src/rag/answer_with_evidence.py` (today's deterministic path) and leaves the model-in-the-loop variant as a deliberate, future step.


## Summary

You compared the two paradigms with the question fixed and the model removed. Once a real LoRA adapter exists, replace the stub in cell 3 with the commented snippet and re-run notebook 08 to measure the lift.


## Exercises

1. Pick three questions where you expect RAG to lose and three where you expect fine-tune to lose. Justify each.
2. Sketch a *hybrid* function `answer_hybrid(g, model, question)` that takes graph evidence and a model handle. What does its prompt look like?
3. Estimate the cost (in tokens and wall-clock) of the hybrid path versus pure RAG.
